# RoPE: позиция как поворот

У одного вектора головы внимания есть несколько **соседних пар координат**. Каждая пара — отдельная плоскость: позиция задаёт поворот, частота пары определяет его скорость. Все индексы начинаются с нуля.

**Запуск:** выберите Python из `.venv` проекта и нажмите **Run All**. Ниже появятся контролы и три связанных представления. Отпустите ползунок, чтобы пересчитать графики. **Сброс к CONFIG** восстанавливает параметры из следующей ячейки.

Это фиксированные синтетические `q` и `k` после проекций одной головы; обучение, веса модели и датасеты не нужны.

In [ ]:
# CONFIG — все исходные параметры в одном месте.
# Изменили CONFIG? Выполните Run All; кнопка сброса использует эти значения.
CONFIG = {
    "d": 8,                       # чётная размерность: 2, 4, …, 32
    "base": 10000.0,              # 10 … 1_000_000, логарифмический ползунок
    "seed": 42,                   # 0 … 2**32−1; воспроизводимые q/k, float64
    "m": 3,                      # позиция q: −128 … 128
    "n": 11,                     # позиция k: −128 … 128
    "c": 0,                      # общий сдвиг: −256 … 256
    "pair": 1,                   # i: 0 … d/2−1; координаты (2i, 2i+1)
    "delta_range": (-64, 64),     # целые Δ для графика: −256 … 256
}

## От координат к углу

Для $i=0,\ldots,d/2-1$ используем
$$\theta_i=\mathrm{base}^{-2i/d},\qquad \alpha_{p,i}=p\theta_i\quad\text{радиан}.$$
Поворот векторов-столбцов против часовой стрелки при положительном угле:
$$R(\alpha)=\begin{pmatrix}\cos\alpha&-\sin\alpha\\\sin\alpha&\cos\alpha\end{pmatrix}.$$
То есть $(x,y)\mapsto(x\cos\alpha-y\sin\alpha,\;x\sin\alpha+y\cos\alpha)$.

В NumPy вектор имеет форму `(d,)`, а пары после `reshape(-1, 2)` хранятся строками. Им соответствует **`pairs @ R(alpha).T`**. В реализации используются эквивалентные покоординатные формулы; порядок пар всегда `(0,1), (2,3), …`. Все вычисления — `float64`.

In [ ]:
from IPython.display import display
from ml_visualizations.rope import rotate, scores
from ml_visualizations.rope_demo import RopeDemo

# Повторное выполнение ячейки закрывает предыдущие виджеты.
if "demo" in globals():
    for widget in [*demo.controls.values(), demo.reset_button,
                   demo.summary, demo.details, demo.image, demo.ui]:
        widget.close()
demo = RopeDemo(CONFIG)
display(demo.ui)

## Как читать представления

- **Плоскость слева — только выбранная пара.** Синий — `q`, оранжевый — `k`; пунктир и пустые квадраты — исходные координаты, сплошные стрелки — после вращения в позициях `m+c`, `n+c`. Масштаб осей одинаковый. Дуги сохраняют знак поворота: плюс — против часовой стрелки, минус — по часовой. Полные обороты опущены; полные знаковые углы и координаты даны под рисунком.
- **Частоты и углы справа — все пары.** Выбранная пара выделена зелёным. При `d=8`, `base=10000` частоты равны $1,0.1,0.01,0.001$: одна позиция поворачивает пары на разные углы. Логарифмическая шкала частот позволяет видеть медленные пары.
- **График снизу — полный dot product, сумма по всем координатам.** `q` и `k` зафиксированы; меняется только относительный сдвиг $\Delta$. Зелёная точка — текущее $n-m$. Если точка вне диапазона, это явно подписано. Таблица под рисунком отдельно показывает вклад каждой пары.

При изменении `d` или `seed` векторы воспроизводимо создаются заново; индекс пары автоматически ограничивается новой размерностью. **Перемещение m, n, c, изменение base и выбор пары не генерируют новые q/k.**

## Почему остаётся относительная позиция

Пусть $R_p$ — блочная матрица всех пар, $q_m=R_mq$ и $k_n=R_nk$. Тогда
$$q_m^\top k_n=q^\top R_m^\top R_nk=q^\top R_{n-m}k.$$
Использованы $R_m^\top=R_{-m}$ и композиция $R_aR_b=R_{a+b}$ для каждой пары. Поэтому общий сдвиг $(m,n)\to(m+c,n+c)$ не меняет **полный score и вклад каждой пары**, пока исходные $q,k$ неизменны. Сами стрелки при этом поворачиваются.

Над графиками рядом показаны три реальных вычисления: прямое, через относительный сдвиг и после общего сдвига. Их расхождения порядка округления выводятся отдельно.

Скалярное произведение — **не вероятность внимания**. В обычном scaled dot-product attention logit равен $q_m^\top k_n/\sqrt d$; вероятность появляется после softmax по ключам (и применения маски, если она нужна). В этой демонстрации показан нескалированный dot product.

In [ ]:
# Минимальная проверка знака; подробные независимые тесты — tests/.
import numpy as np
reference = scores([1.0, 0.0], [0.0, 1.0], m=0, n=1)
np.testing.assert_allclose(list(reference.values()), -np.sin(1.0), rtol=1e-11, atol=1e-12)
np.testing.assert_array_equal(rotate(demo.q, 0), demo.q)
print(f"Эталон d=2: q₀ᵀk₁ = {reference['direct']:.9f} = −sin(1). Проверка пройдена.")

## Границы демонстрации

Здесь стандартный RoPE без масштабирования контекста, с полным вращением чётной размерности. Один фиксированный синтетический `q` и `k` не описывает распределение векторов настоящей модели. **Конкретный score не обязан монотонно уменьшаться с расстоянием**: разные пары дают колеблющиеся вклады.

Исходники опубликованы без outputs. GitHub показывает код и текст, но не запускает Python: для контролов нужен локальный живой kernel. Проверки разработчика: `python -m pytest -q` и `python scripts/verify_notebook.py`; выполненная копия сохраняется локально в ignored `exports/`.

**Один шаг для исследования:** оставьте `q/k`, `m` и `n` фиксированными и меняйте только `c`. Сопоставьте поворот стрелок с тремя числами полного score.

Первоисточник: [Su et al., RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864).